In [ ]:
# Core libs for file handling, arrays, dataframes, and plotting
import os  # filesystem paths and directory utilities
import numpy as np  # vectorized numerical ops
import pandas as pd  # tabular data handling
import seaborn as sns  # statistical plotting
sns.set_style('darkgrid')  # consistent dark grid background for plots
import matplotlib.pyplot as plt  # plotting primitives
from sklearn.model_selection import train_test_split  # dataset splitting helper
from sklearn.metrics import confusion_matrix, classification_report  # evaluation metrics

from PIL import Image  # image I/O and basic manipulation

# Deep learning stack (TensorFlow/Keras) and computer vision helpers
import tensorflow as tf  # core TensorFlow engine
from tensorflow import keras  # Keras high-level API
from tensorflow.keras.models import Sequential  # linear stack of layers
from tensorflow.keras.optimizers import Adam, Adamax  # optimizers
from tensorflow.keras.metrics import categorical_crossentropy  # loss metric alias
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # augmentation pipeline
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization  # layer building blocks
from tensorflow.keras import regularizers  # weight regularization
import cv2  # OpenCV for image processing
import warnings  # control warnings
from tqdm import tqdm  # progress bars

# Silence noisy warnings to keep notebook output clean
warnings.filterwarnings("ignore")

print('modules loaded')  # quick sanity check that imports succeeded

## Dataset locations
Define dataset root paths for train/val/test splits.

In [ ]:
# Define base dataset folder and split-specific paths
dataset_path = "dataset"  # root folder holding train/val/test

train_path = os.path.join(dataset_path, "train")  # training images
val_path = os.path.join(dataset_path, "val")  # validation images
test_path = os.path.join(dataset_path, "test")  # test images

## Re-split dataset to 70/20/10
Regenerate balanced train/val/test folders under `dataset_70_20_10`.

In [ ]:
# Re-split existing dataset into 70/20/10 while preserving class balance
from pathlib import Path
import shutil

source_root = Path("dataset")
target_root = Path("dataset_70_20_10")

# start clean so counts don't accumulate on reruns
if target_root.exists():
    shutil.rmtree(target_root)

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

# collect all images per class across current splits
class_dirs = sorted(p.name for p in (source_root / "train").iterdir() if p.is_dir())
all_images = {cls: [] for cls in class_dirs}
for split in ["train", "val", "test"]:
    for cls in class_dirs:
        for img_path in (source_root / split / cls).glob("*"):
            if img_path.is_file():
                all_images[cls].append(img_path)

# split 70/20/10 using train_test_split for reproducibility
new_counts = {"train": {}, "val": {}, "test": {}}
for cls, imgs in all_images.items():
    train_imgs, tmp_imgs = train_test_split(imgs, test_size=0.3, random_state=42, shuffle=True)
    val_imgs, test_imgs = train_test_split(tmp_imgs, test_size=2/3, random_state=42, shuffle=True)
    split_map = {"train": train_imgs, "val": val_imgs, "test": test_imgs}

    for split, split_imgs in split_map.items():
        dest_dir = target_root / split / cls
        ensure_dir(dest_dir)
        for src in split_imgs:
            shutil.copy2(src, dest_dir / src.name)
        new_counts[split][cls] = len(split_imgs)

print("Re-split complete ->", target_root)
print("Counts per split:")
for split in ["train", "val", "test"]:
    print(split, new_counts[split])

# Point the rest of the notebook to the new split
dataset_path = str(target_root)
train_path = os.path.join(dataset_path, "train")
val_path = os.path.join(dataset_path, "val")
test_path = os.path.join(dataset_path, "test")

## Class list and counts
List class folders, tally images per split, and sanity-check balance.

In [ ]:
# Inspect available class folders under the training split
classes = os.listdir(train_path)  # list class folder names inside train
print("Classes:", classes)  # show class labels detected

In [ ]:
# Helper to count images per class inside a split folder
def count_images(folder):
    counts = {}  # accumulator for {class: count}
    for cls in os.listdir(folder):  # iterate class folders
        cls_path = os.path.join(folder, cls)  # full path to class dir
        counts[cls] = len(os.listdir(cls_path))  # number of files in class dir
    return counts

# Tally image counts across splits to check balance
train_counts = count_images(train_path)
val_counts = count_images(val_path)
test_counts = count_images(test_path)

print("Train:", train_counts)
print("Validation:", val_counts)
print("Test:", test_counts)

## Visualize training distribution
Bar chart of class counts in the training split.

In [ ]:
# Visualize class distribution in the training split
plt.figure(figsize=(8,5))  # set canvas size
sns.barplot(x=list(train_counts.keys()), y=list(train_counts.values()))  # bars for each class count
plt.title("Training Data Distribution")  # chart title
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=30)  # tilt labels for readability
plt.show()

In [ ]:
# Build a dataframe listing every image with its split and label
records = []  # accumulator rows

for split in ['train','val','test']:
    split_path = os.path.join(dataset_path, split)  # current split folder

    for label in os.listdir(split_path):
        class_path = os.path.join(split_path, label)  # class folder path

        for img in os.listdir(class_path):
            records.append({
                "split": split,  # which split this file belongs to
                "label": label,  # class name
                "filepath": os.path.join(class_path, img)  # full file path
            })

eda_df = pd.DataFrame(records)  # assemble rows into dataframe

print("Total images:", len(eda_df))
eda_df.head()  # preview first rows

## Build EDA dataframe
Enumerate all files with split/label metadata for downstream analysis.

In [ ]:
# Sample image dimensions to understand typical sizes
sizes = []  # holds (width, height) tuples

sample_paths = eda_df['filepath'].sample(200)  # random subset of images

for path in sample_paths:
    img = Image.open(path)  # open image
    sizes.append(img.size)  # record (width, height)

size_df = pd.DataFrame(sizes, columns=['width','height'])  # to analyze size distribution

print(size_df.describe())  # summary stats for width/height

## Image dimension sampling
Sample image sizes to choose a sensible target resize.

In [ ]:
# Inspect pixel value range on a sample image
sample_img = np.array(Image.open(eda_df['filepath'].iloc[0]))  # load first image into array

print("Min pixel value:", sample_img.min())  # darkest pixel
print("Max pixel value:", sample_img.max())  # brightest pixel

## Pixel range sanity check
Confirm images are in expected 0–255 range.

In [ ]:
# Show a few example images per class to spot visual patterns
samples_per_class = 3  # images per label to display

for label in classes:
    class_path = os.path.join(train_path, label)  # folder for this label

    images = os.listdir(class_path)[:samples_per_class]  # pick first few files

    plt.figure(figsize=(10,3))  # one row of subplots

    for i, img_name in enumerate(images):
        img = cv2.imread(os.path.join(class_path, img_name))  # load with OpenCV (BGR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # convert to RGB for plotting

        plt.subplot(1,3,i+1)  # select subplot slot
        plt.imshow(img)
        plt.title(label)
        plt.axis("off")  # hide axes for clarity

    plt.show()

## Visual samples per class
Show a few images from each class to spot issues or patterns.

In [ ]:
# Check for unreadable/corrupted images
broken = []  # collect paths that fail to open

for path in eda_df['filepath']:
    try:
        Image.open(path)  # attempt to open
    except:
        broken.append(path)  # record failures

print("Broken images:", len(broken))

## Corruption check
Attempt to open every file and flag unreadable images.

In [ ]:
# Build a sampled stats dataframe capturing geometry, color, and texture features per image
tqdm.pandas()  # enable pandas integration for progress bars

# Sample a manageable subset to compute stats
stat_sample = eda_df.sample(min(len(eda_df), 600), random_state=42).reset_index(drop=True)  # limit to 600 for speed
stat_records = []  # feature rows

for _, row in tqdm(stat_sample.iterrows(), total=len(stat_sample)):
    img = Image.open(row['filepath']).convert('RGB')  # read image as RGB
    arr = np.array(img)  # to numpy
    h, w, _ = arr.shape  # spatial dims
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)  # grayscale version
    hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)  # HSV for brightness/saturation
    sobel = cv2.Sobel(gray, cv2.CV_64F, 1, 1, ksize=3)  # edge response

    # Aggregate per-image stats for later plots
    stat_records.append({
        'split': row['split'],  # which split
        'label': row['label'],  # class
        'width': w,
        'height': h,
        'area': w * h,  # pixel count
        'aspect': w / h,  # width/height ratio
        'orientation': 'landscape' if w > h else ('portrait' if h > w else 'square'),
        'mean_r': arr[:, :, 0].mean(),  # average red
        'mean_g': arr[:, :, 1].mean(),  # average green
        'mean_b': arr[:, :, 2].mean(),  # average blue
        'brightness_v': hsv[:, :, 2].mean(),  # value channel mean
        'saturation_s': hsv[:, :, 1].mean(),  # saturation channel mean
        'contrast_gray': gray.std(),  # grayscale std as contrast proxy
        'edge_strength': np.mean(np.abs(sobel)),  # mean absolute edge magnitude
        'file_kb': os.path.getsize(row['filepath']) / 1024  # file size in KB
    })

stat_df = pd.DataFrame(stat_records)  # assemble feature table
stat_df['resolution'] = stat_df['width'].astype(str) + 'x' + stat_df['height'].astype(str)  # human-readable size

print(stat_df.head())
print('\nSample size used:', len(stat_df))

## Image-level feature extraction
Compute geometry, color, and texture stats on a sampled subset for downstream visuals.

**Why Visual 1 & 2 (count plots)**: Bar counts reveal whether splits or labels are imbalanced. They use seaborn `countplot` to tally rows from `eda_df`, showing split and class frequencies. Without these, we could miss data imbalance that skews training/validation and metrics.

In [ ]:
# Visual 1: overall split distribution
plt.figure(figsize=(5,4))  # compact bar chart
sns.countplot(data=eda_df, x='split', order=['train','val','test'], palette='Set2')  # bars per split
plt.title('Images per split')
plt.xlabel('Split')
plt.ylabel('Count')
plt.show()

# Visual 2: overall class distribution across all splits
plt.figure(figsize=(6,4))
sns.countplot(data=eda_df, x='label', order=sorted(eda_df['label'].unique()), palette='Set3')  # bars per label
plt.title('Images per class (all splits)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=20)  # tilt labels for readability
plt.show()

## Split and class distributions
Count plots for splits and labels to spot imbalance.

**Why Visual 3 & 4 (stacked/grouped bars)**: These compare class counts across train/val/test in one view using the precomputed `counts` table. Stacked shows total volume; grouped highlights per-split differences. Skipping them hides split-specific imbalance that can cause biased evaluation or sampling issues.

In [ ]:
# Compare class counts per split using stacked and grouped bars
counts = eda_df.groupby(['split','label']).size().unstack(fill_value=0)[sorted(eda_df['label'].unique())]  # pivot counts
counts.loc[['train','val','test']].plot(kind='bar', stacked=True, figsize=(7,4), colormap='tab20')  # stacked view
plt.title('Class distribution per split (stacked)')
plt.xlabel('Split')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Label')
plt.show()

counts.loc[['train','val','test']].plot(kind='bar', stacked=False, figsize=(7,4), colormap='Paired')  # grouped bars
plt.title('Class distribution per split (grouped)')
plt.xlabel('Split')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Label')
plt.show()

## Split/class comparison
Stacked and grouped bars to compare class counts across splits.

**Why Visual 5 (heatmap)**: Heatmap of the `counts` matrix quickly spotlights over/under-represented class-split combos with color intensity. Without it, we rely on numbers alone and may overlook sparse cells that can cause poor generalization for specific classes in a split.

In [ ]:
# Heatmap to highlight count imbalances across split/label
plt.figure(figsize=(6,4))
sns.heatmap(counts.loc[['train','val','test']], annot=True, fmt='d', cmap='YlGnBu')  # color-coded counts
plt.title('Visualizing Heatmap of class counts by split')
plt.xlabel('Label')
plt.ylabel('Split')
plt.show()

## Imbalance heatmap
Heatmap of class counts per split to highlight sparse cells.

**Why Visual 6 & 7 (pie charts)**: Pies summarize proportion of labels overall and splits overall, emphasizing share instead of counts. They make imbalance intuitive at a glance. Omitting them removes an easy percentage view; you’d rely on bar heights only, which can hide proportion differences when totals vary.

In [ ]:
# Pie charts summarizing label and split proportions
plt.figure(figsize=(5,5))
eda_df['label'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('Set3'))  # label share
plt.ylabel('')
plt.title('Label share (all splits)')
plt.show()

plt.figure(figsize=(4.5,4.5))
eda_df['split'].value_counts().reindex(['train','val','test']).plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('Set2'))  # split share
plt.ylabel('')
plt.title('Split share')
plt.show()

## Proportion pies
Pie charts to show label and split share as percentages.

**Why Visual 8 & 9 (width/height histograms)**: Histograms of dimensions show common image sizes and spread, guiding resize strategy. Without them we might choose an input size that crops or stretches many images, hurting model quality.

In [ ]:
# Width and height distributions from sampled stats
plt.figure(figsize=(6,4))
sns.histplot(stat_df['width'], bins=30, color='steelblue', kde=True)  # histogram of widths
plt.title('Width distribution (sample)')
plt.xlabel('Width (px)')
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(stat_df['height'], bins=30, color='salmon', kde=True)  # histogram of heights
plt.title('Height distribution (sample)')
plt.xlabel('Height (px)')
plt.ylabel('Frequency')
plt.show()

## Geometry histograms
Width and height distributions from sampled images.

**Why Visual 10 & 11 (hexbin + aspect histogram)**: Hexbin shows joint density of width vs height to spot typical resolutions and outliers; the aspect histogram shows shape tendencies. Skipping them hides whether sizes cluster along certain diagonals or if extreme aspect ratios need special handling.

In [ ]:
# Joint view of width vs height plus aspect ratio distribution
plt.figure(figsize=(6,5))
plt.hexbin(stat_df['width'], stat_df['height'], gridsize=25, cmap='viridis')  # 2D density of sizes
plt.title('Width vs Height density')
plt.xlabel('Width (px)')
plt.ylabel('Height (px)')
cb = plt.colorbar()
cb.set_label('Count')
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(stat_df['aspect'], bins=30, color='mediumseagreen', kde=True)  # aspect ratio distribution
plt.title('Aspect ratio (W/H) distribution')
plt.xlabel('Aspect ratio')
plt.ylabel('Frequency')
plt.show()

## Size density and aspect ratios
Hexbin of width/height and histogram of aspect ratios.

**Why Visual 12–14 (boxplots width/height/aspect by label)**: Boxplots compare geometry across classes to detect if one class has systematically different image sizes or shapes. Without them, class-specific preprocessing needs could be missed, leading to distortions for certain labels.

In [ ]:
# Boxplots to compare geometry across labels
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='width', order=sorted(stat_df['label'].unique()), palette='Pastel1')  # width per class
plt.title('Width by label (sample)')
plt.xlabel('Label')
plt.ylabel('Width (px)')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='height', order=sorted(stat_df['label'].unique()), palette='Pastel2')  # height per class
plt.title('Height by label (sample)')
plt.xlabel('Label')
plt.ylabel('Height (px)')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='aspect', order=sorted(stat_df['label'].unique()), palette='Set2')  # aspect ratio per class
plt.title('Aspect ratio by label (sample)')
plt.xlabel('Label')
plt.ylabel('Aspect ratio (W/H)')
plt.xticks(rotation=20)
plt.show()

## Geometry by label
Boxplots comparing width, height, and aspect ratio across classes.

**Why Visual 15 & 16 (orientation counts, area histogram)**: Orientation counts reveal dominant landscape/portrait mix, informing augmentation strategy; area histogram shows pixel count scale. Skipping them risks choosing crops/resize policies that penalize minority orientations or overwhelm memory with very large images.

In [ ]:
# Orientation mix and overall pixel area distribution
plt.figure(figsize=(5,4))
sns.countplot(data=stat_df, x='orientation', order=['landscape','portrait','square'], palette='coolwarm')  # orientation counts
plt.title('Orientation distribution (sample)')
plt.xlabel('Orientation')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(stat_df['area'], bins=30, color='mediumpurple', kde=True)  # pixel area histogram
plt.title('Image area distribution (sample)')
plt.xlabel('Area (px^2)')
plt.ylabel('Frequency')
plt.show()

## Orientation and area
Orientation mix and pixel-area distribution to guide resizing/augmentation.

**Why Visual 17 (grayscale histogram)**: Sampling grayscale pixels shows contrast and brightness spread, indicating if normalization or CLAHE might help. Without it, we might miss low-contrast images that hinder edge detectors and CNN learning.

In [ ]:
# Sample pixel intensities to see grayscale distribution
plt.figure(figsize=(6,4))
gray_values = []  # sampled grayscale pixels
for path in stat_sample['filepath'][:200]:  # limit to avoid huge arrays
    g = np.array(Image.open(path).convert('L')).flatten()  # grayscale pixels
    gray_values.extend(g[np.random.choice(len(g), size=min(2000, len(g)), replace=False)])  # random subset per image
sns.histplot(gray_values, bins=50, color='gray', kde=True)
plt.title('Grayscale intensity distribution (sample of pixels)')
plt.xlabel('Intensity (0-255)')
plt.ylabel('Frequency')
plt.show()

## Grayscale intensity
Histogram of sampled grayscale pixels to assess contrast/brightness spread.

**Why Visual 18 (mean RGB histograms)**: Shows distribution of per-image channel means, revealing color cast or illumination bias. Skipping it hides whether a color normalization step (e.g., per-channel standardization) is needed to stabilize training.

In [ ]:
# Distribution of per-image mean RGB channels
plt.figure(figsize=(7,4))
channel_colors = ['red','green','blue']
for ch, color in enumerate(channel_colors):
    sns.histplot(stat_df[[f'mean_{c}' for c in ['r','g','b']][ch]], bins=30, color=color, label=f'{color} mean', alpha=0.5)  # channel mean histogram
plt.title('Mean RGB values per image (sample)')
plt.xlabel('Channel mean (0-255)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## Mean RGB distribution
Channel-mean histograms to spot global color casts.

**Why Visual 19 (mean RGB per class)**: Class-level barplot checks if certain labels have systematic color differences, which might be discriminative or require balancing. Without it, we could miss color leakage cues or fail to apply augmentations that reduce over-reliance on color.

In [ ]:
# Compare average RGB levels across classes
rgb_means = stat_df.groupby('label')[['mean_r','mean_g','mean_b']].mean().reset_index()  # per-class averages
rgb_melt = rgb_means.melt(id_vars='label', var_name='channel', value_name='mean_val')  # long format for seaborn
plt.figure(figsize=(7,4))
sns.barplot(data=rgb_melt, x='label', y='mean_val', hue='channel', palette={'mean_r':'red','mean_g':'green','mean_b':'blue'})
plt.title('Mean RGB per class (sample)')
plt.xlabel('Label')
plt.ylabel('Channel mean (0-255)')
plt.xticks(rotation=20)
plt.legend(title='Channel')
plt.show()

## Per-class color means
Barplot of mean RGB per class to detect color biases.

**Why Visual 20–23 (brightness/saturation/contrast/edges by label)**: Boxplots of V, S, gray std, and Sobel edge strength highlight per-class texture/lightness differences. Skipping them hides whether some classes are darker, less saturated, or smoother—factors that can cause biased feature learning or suggest preprocessing (e.g., histogram equalization, sharpening).

In [ ]:
# Compare brightness, saturation, contrast, and edge strength per class
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='brightness_v', order=sorted(stat_df['label'].unique()), palette='YlOrBr')  # V channel per class
plt.title('Brightness (V channel) by label (sample)')
plt.xlabel('Label')
plt.ylabel('Brightness (0-255)')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='saturation_s', order=sorted(stat_df['label'].unique()), palette='PuBuGn')  # saturation per class
plt.title('Saturation (S channel) by label (sample)')
plt.xlabel('Label')
plt.ylabel('Saturation (0-255)')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='contrast_gray', order=sorted(stat_df['label'].unique()), palette='cool')  # grayscale contrast per class
plt.title('Contrast (gray std) by label (sample)')
plt.xlabel('Label')
plt.ylabel('Std of gray')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='edge_strength', order=sorted(stat_df['label'].unique()), palette='rocket')  # edge strength per class
plt.title('Edge strength by label (sample)')
plt.xlabel('Label')
plt.ylabel('Mean abs Sobel response')
plt.xticks(rotation=20)
plt.show()

## Brightness, saturation, texture by label
Boxplots for V/S channels, grayscale contrast, and edge strength across classes.

**Why Visual 24 & 25 (correlation heatmap, top resolutions)**: Correlation map shows relationships among numeric features (size, color, texture), guiding feature selection or decorrelation. Top-resolution bar plot surfaces the most common shapes to pick efficient resizing targets. Without these, we might choose redundant features or an inefficient canonical resolution, wasting compute or losing detail.

In [ ]:
# Correlation across numeric features and most common resolutions
plt.figure(figsize=(8,6))
num_cols = ['width','height','area','aspect','mean_r','mean_g','mean_b','brightness_v','saturation_s','contrast_gray','edge_strength','file_kb']  # numeric feature list
corr = stat_df[num_cols].corr()  # correlation matrix
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)  # visualize correlations
plt.title('Correlation of image-level features (sample)')
plt.show()

plt.figure(figsize=(7,4))
stat_df['resolution'].value_counts().head(10).plot(kind='bar', color='teal')  # top resolutions
plt.title('Top 10 resolutions (sample)')
plt.xlabel('Resolution (WxH)')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.show()

## Feature correlations and common resolutions
Correlation heatmap of numeric features plus most frequent resolutions.

In [ ]:
# Image preprocessing: build train/val/test generators from active split
from pathlib import Path

action_root = Path("dataset_70_20_10") if Path("dataset_70_20_10").exists() else Path(dataset_path)
train_dir = action_root / "train"
val_dir = action_root / "val"
test_dir = action_root / "test"

img_size = (224, 224)
batch_size = 32

augment = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode="nearest",
)
plain = ImageDataGenerator(rescale=1.0/255)

train_ds = augment.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True,
)
val_ds = plain.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False,
)
test_ds = plain.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False,
)

num_classes = len(train_ds.class_indices)
input_shape = img_size + (3,)
print("Generators ready from", action_root)

## Model training utilities
Shared helpers to compile, train, evaluate, and plot metrics for all architectures below.

In [ ]:
# Keras training helper (shared across EfficientNet, ViT, and additional models)
from tensorflow.keras.applications import EfficientNetB0, DenseNet121, VGG16, InceptionV3, MobileNetV2, ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Simple utility to compile, train, and evaluate a Keras model
results = {}

def compile_and_train(model, label: str, epochs: int = 20):
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        verbose=1,
    )
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    print(f"{label} -> test_acc={test_acc:.3f}, test_loss={test_loss:.3f}")
    results[label] = {"history": history.history, "test_acc": test_acc, "test_loss": test_loss}
    return results[label]


In [ ]:
# Utility: plot training curves for a model in `results`

def plot_history(label: str, title: str = None):
    if label not in results or "history" not in results[label]:
        print(f"No history available for {label}; run training first.")
        return
    hist = results[label]["history"]
    acc = hist.get("accuracy", hist.get("acc", []))
    val_acc = hist.get("val_accuracy", hist.get("val_acc", []))
    loss = hist.get("loss", [])
    val_loss = hist.get("val_loss", [])
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="train_acc")
    plt.plot(epochs_range, val_acc, label="val_acc")
    plt.title(title or f"{label}: Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="train_loss")
    plt.plot(epochs_range, val_loss, label="val_loss")
    plt.title(title or f"{label}: Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.suptitle(title or label)
    plt.tight_layout()
    plt.show()


## CNN baseline
Build, train, plot, and save a simple convolutional baseline to benchmark transfer models.

In [ ]:
# Define CNN baseline

# Builds a shallow convolutional classifier with batch norm regularization.
def make_cnn_baseline():
    model = Sequential(name="CNN_baseline")
    model.add(Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())
    model.add(Flatten())
    model.add(Dense(256, activation="relu"))
    model.add(Dropout(0.4))
    model.add(Dense(num_classes, activation="softmax"))
    return model


### CNN baseline — define architecture
Construct a small CNN with batch norm and dropout to act as a baseline classifier.

In [ ]:
# Train CNN baseline
# Set placeholder epochs; increase later for real training.
epochs = 1
cnn_model = make_cnn_baseline()
cnn_results = compile_and_train(cnn_model, "cnn_baseline", epochs=epochs)

# Quick view of accumulated test metrics for all trained models.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### CNN baseline — train and evaluate
Instantiate the CNN, train for a placeholder epoch, and log the test metrics.

In [ ]:
# Plot CNN baseline training curves
# Uses shared helper to draw accuracy/loss for this run.
plot_history("cnn_baseline", title="CNN Baseline")


### CNN baseline — plot learning curves
Visualize train/val accuracy and loss for the CNN baseline.

In [ ]:
# Save CNN baseline model
from pathlib import Path

# Ensure models directory exists then save the baseline weights.
Path("models").mkdir(exist_ok=True)
cnn_model.save("models/cnn_baseline.h5")
print("Saved CNN baseline to models/cnn_baseline.h5")


### CNN baseline — save weights
Persist the trained CNN baseline to disk for later use.

## EfficientNetB0
Freeze the ImageNet backbone, add a small head, then train, plot curves, and save the model.

In [ ]:
# Define EfficientNetB0

# Use ImageNet-pretrained EfficientNetB0 as frozen feature extractor with a custom head.
def make_efficientnet_b0():
    base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="EfficientNetB0_frozen")


### EfficientNetB0 — define architecture
Freeze EfficientNetB0 backbone and attach a small dense head for classification.

In [ ]:
# Train EfficientNetB0
# Set placeholder epochs; adjust upward for full training.
epochs = 1
EffNetB0_model = make_efficientnet_b0()
eff_results = compile_and_train(EffNetB0_model, "efficientnet_b0", epochs=epochs)

# Show cumulative test metrics from all runs so far.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### EfficientNetB0 — train and evaluate
Instantiate EfficientNetB0 classifier, run placeholder epoch, and log metrics.

In [ ]:
# Plot EfficientNetB0 training curves
# Helper plots accuracy and loss across epochs.
plot_history("efficientnet_b0", title="EfficientNetB0")


### EfficientNetB0 — plot learning curves
Use shared plotting helper to visualize EfficientNetB0 training history.

## DenseNet121
Use a frozen DenseNet121 feature extractor, fine-tune the classifier head, visualize training, and save.

In [ ]:
# Define DenseNet121

# Use ImageNet-pretrained DenseNet121 frozen as feature extractor with custom classifier head.
def make_densenet121():
    base = DenseNet121(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="DenseNet121_frozen")


### DenseNet121 — define architecture
Freeze DenseNet121 features and attach a compact classifier head.

In [ ]:
# Train DenseNet121
# Placeholder epochs value; increase for full runs.
epochs = 1
densenet_model = make_densenet121()
densenet_results = compile_and_train(densenet_model, "densenet121", epochs=epochs)

# Display summary metrics accumulated so far.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### DenseNet121 — train and evaluate
Create the DenseNet121 model, run one epoch placeholder training, and record metrics.

In [ ]:
# Plot DenseNet121 training curves
# Visualize training and validation metrics for DenseNet121.
plot_history("densenet121", title="DenseNet121")


### DenseNet121 — plot learning curves
Plot accuracy and loss for the DenseNet121 run via the shared helper.

In [ ]:
# Save DenseNet121 model
from pathlib import Path

# Persist the trained DenseNet121 weights.
Path("models").mkdir(exist_ok=True)
densenet_model.save("models/densenet121.h5")
print("Saved DenseNet121 to models/densenet121.h5")


### DenseNet121 — save weights
Store the DenseNet121 model after training.

## VGG16
Train a frozen VGG16 backbone with a lightweight head, plot accuracy/loss, and persist the weights.

In [ ]:
# Define VGG16

# Use pretrained VGG16 frozen as a feature extractor with a small dense head.
def make_vgg16():
    base = VGG16(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="VGG16_frozen")


### VGG16 — define architecture
Freeze VGG16 convolutional base and attach a custom dense classifier.

In [ ]:
# Train VGG16
# Placeholder epochs value; increase when ready to train longer.
epochs = 1
vgg16_model = make_vgg16()
vgg16_results = compile_and_train(vgg16_model, "vgg16", epochs=epochs)

# Report metrics across all trained models.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### VGG16 — train and evaluate
Instantiate VGG16 classifier, run placeholder epoch, and capture metrics.

In [ ]:
# Plot VGG16 training curves
# Visualize training/validation accuracy and loss for VGG16.
plot_history("vgg16", title="VGG16")


### VGG16 — plot learning curves
Plot train/val accuracy and loss for the VGG16 run.

In [ ]:
# Save VGG16 model
from pathlib import Path

# Persist VGG16 weights for later inference.
Path("models").mkdir(exist_ok=True)
vgg16_model.save("models/vgg16.h5")
print("Saved VGG16 to models/vgg16.h5")


### VGG16 — save weights
Save the trained VGG16 model to disk.

## InceptionV3
Leverage InceptionV3 features with a custom head, track training curves, and save the model.

In [ ]:
# Define InceptionV3

# Use pretrained InceptionV3 frozen as feature extractor with a dense head.
def make_inception_v3():
    base = InceptionV3(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="InceptionV3_frozen")


### InceptionV3 — define architecture
Freeze InceptionV3 base and add a compact classifier head.

In [ ]:
# Train InceptionV3
# Placeholder epochs value; increase later for better convergence.
epochs = 1
inception_model = make_inception_v3()
inception_results = compile_and_train(inception_model, "inception_v3", epochs=epochs)

# Display current metrics table.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### InceptionV3 — train and evaluate
Instantiate InceptionV3 classifier, run one-epoch placeholder training, and print metrics.

In [ ]:
# Plot InceptionV3 training curves
# Visualize train/validation metrics for InceptionV3.
plot_history("inception_v3", title="InceptionV3")


### InceptionV3 — plot learning curves
Plot accuracy/loss for the InceptionV3 run using the shared helper.

In [ ]:
# Save InceptionV3 model
from pathlib import Path

# Persist trained InceptionV3 weights.
Path("models").mkdir(exist_ok=True)
inception_model.save("models/inception_v3.h5")
print("Saved InceptionV3 to models/inception_v3.h5")


### InceptionV3 — save weights
Save the trained InceptionV3 model to disk.

## MobileNetV2
Train a lightweight MobileNetV2 classifier head, inspect learning curves, and save for reuse.

In [ ]:
# Define MobileNetV2

# Use pretrained MobileNetV2 frozen as feature extractor with a dense classification head.
def make_mobilenet_v2():
    base = MobileNetV2(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="MobileNetV2_frozen")


### MobileNetV2 — define architecture
Freeze MobileNetV2 backbone and attach a lightweight dense head.

## ResNet50
Train and evaluate a frozen ResNet50 backbone with a dense head, plot performance, and save the artifact.

### ResNet50 — define architecture
Freeze ResNet50 backbone and add a dense head for classification.

In [ ]:
# Define ResNet50

# Use pretrained ResNet50 frozen as feature extractor with a dense head.
def make_resnet50():
    base = ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="ResNet50_frozen")


In [ ]:
# Train MobileNetV2
# Placeholder epochs; increase for proper training.
epochs = 1
mobilenet_model = make_mobilenet_v2()
mobilenet_results = compile_and_train(mobilenet_model, "mobilenet_v2", epochs=epochs)

# Print consolidated metrics across models.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### MobileNetV2 — train and evaluate
Instantiate MobileNetV2 classifier, run one epoch, and show metrics.

In [ ]:
# Train ResNet50
# Placeholder epochs value; bump up when ready for longer training.
epochs = 1
resnet_model = make_resnet50()
resnet_results = compile_and_train(resnet_model, "resnet50", epochs=epochs)

# Display cumulative metrics across models.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### ResNet50 — train and evaluate
Instantiate ResNet50 classifier, run one epoch placeholder, and log metrics.

In [ ]:
# Plot MobileNetV2 training curves
# Visualize MobileNetV2 accuracy and loss progression.
plot_history("mobilenet_v2", title="MobileNetV2")


### MobileNetV2 — plot learning curves
Plot training/validation accuracy and loss for MobileNetV2.

In [ ]:
# Save MobileNetV2 model
from pathlib import Path

# Persist MobileNetV2 weights for reuse.
Path("models").mkdir(exist_ok=True)
mobilenet_model.save("models/mobilenet_v2.h5")
print("Saved MobileNetV2 to models/mobilenet_v2.h5")


### MobileNetV2 — save weights
Save the trained MobileNetV2 model to disk.

In [ ]:
# Plot ResNet50 training curves
# Visualize training/validation metrics for ResNet50.
plot_history("resnet50", title="ResNet50")


### ResNet50 — plot learning curves
Plot accuracy and loss for the ResNet50 run using the helper.

In [ ]:
# Visualize EfficientNetB0 training curves
# Manual plotting (redundant with plot_history) for flexibility.
if "efficientnet_b0" in results and "history" in results["efficientnet_b0"]:
    hist = results["efficientnet_b0"]["history"]
    acc = hist.get("accuracy", hist.get("acc", []))
    val_acc = hist.get("val_accuracy", hist.get("val_acc", []))
    loss = hist.get("loss", [])
    val_loss = hist.get("val_loss", [])
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="train_acc")
    plt.plot(epochs_range, val_acc, label="val_acc")
    plt.title("EfficientNetB0: Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="train_loss")
    plt.plot(epochs_range, val_loss, label="val_loss")
    plt.title("EfficientNetB0: Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.suptitle("Training curves: EfficientNetB0")
    plt.tight_layout()
    plt.show()
else:
    print("No history available for EfficientNetB0; run training first.")


### EfficientNetB0 — manual plot (optional)
Manual Matplotlib plotting of EfficientNetB0 accuracy/loss (redundant with helper above).

In [ ]:
# Save EfficientNetB0 Keras model
from pathlib import Path

# Persist EfficientNetB0 weights.
Path("models").mkdir(exist_ok=True)
EffNetB0_model.save("models/efficientnet_b0.h5")
print("Saved EfficientNetB0 to models/efficientnet_b0.h5")


### EfficientNetB0 — save weights
Save the trained EfficientNetB0 model to disk.

In [ ]:
# Save ResNet50 model
from pathlib import Path

# Persist ResNet50 weights for later inference.
Path("models").mkdir(exist_ok=True)
resnet_model.save("models/resnet50.h5")
print("Saved ResNet50 to models/resnet50.h5")


### ResNet50 — save weights
Save the trained ResNet50 model to disk.

## Vision Transformer (Keras)
Compact ViT classifier: define the transformer, train/evaluate, plot learning curves, and save the weights.

In [ ]:
# Define Vision Transformer (Keras)
import tensorflow as tf
from tensorflow.keras import layers, Model

# Hyperparameters for a compact ViT
patch_size = 16
projection_dim = 64
transformer_layers = 6
num_heads = 4
mlp_units = [128, 64]
dropout_rate = 0.1

# Number of patches after splitting the image.
num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)


def mlp(x, hidden_units, dropout):
    # Lightweight MLP block used inside the transformer.
    for units in hidden_units:
        x = layers.Dense(units, activation="gelu")(x)
        x = layers.Dropout(dropout)(x)
    return x


def build_vit_classifier():
    # Patchify input and project to embedding dimension.
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(filters=projection_dim, kernel_size=patch_size, strides=patch_size, padding="valid")(inputs)
    x = layers.Reshape((num_patches, projection_dim))(x)

    # Add positional embeddings.
    positions = tf.range(start=0, limit=num_patches, delta=1)
    pos_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)(positions)
    x = x + pos_embedding

    # Transformer encoder blocks.
    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate)(x1, x1)
        x2 = layers.Add()([attn_output, x])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp(x3, hidden_units=[projection_dim * 2, projection_dim], dropout=dropout_rate)
        x = layers.Add()([x3, x2])

    # Classification head.
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs=inputs, outputs=outputs, name="ViT_Keras")


### Vision Transformer — define architecture
Build a compact ViT: patch embedding, transformer blocks, and classifier head.

In [ ]:
# Train Vision Transformer (Keras)
# Placeholder epochs; raise later for better performance.
epochs = 1
vit_keras_model = build_vit_classifier()
vit_results = compile_and_train(vit_keras_model, "vit_keras", epochs=epochs)

# Report metrics across all models trained so far.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### Vision Transformer — train and evaluate
Instantiate the ViT model, run one placeholder epoch, and print metrics.

In [ ]:
# Visualize ViT (Keras) training curves
# Manual Matplotlib plot of accuracy/loss for ViT.
if "vit_keras" in results and "history" in results["vit_keras"]:
    hist = results["vit_keras"]["history"]
    acc = hist.get("accuracy", hist.get("acc", []))
    val_acc = hist.get("val_accuracy", hist.get("val_acc", []))
    loss = hist.get("loss", [])
    val_loss = hist.get("val_loss", [])
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="train_acc")
    plt.plot(epochs_range, val_acc, label="val_acc")
    plt.title("ViT (Keras): Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="train_loss")
    plt.plot(epochs_range, val_loss, label="val_loss")
    plt.title("ViT (Keras): Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.suptitle("Training curves: ViT (Keras)")
    plt.tight_layout()
    plt.show()
else:
    print("No history available for ViT (Keras); run training first.")


### Vision Transformer — plot learning curves
Plot training and validation accuracy/loss for the ViT run.

In [ ]:
# Save Keras ViT model
from pathlib import Path

# Persist ViT weights.
Path("models").mkdir(exist_ok=True)
vit_keras_model.save("models/vit_keras.h5")
print("Saved ViT (Keras) to models/vit_keras.h5")


### Vision Transformer — save weights
Save the trained ViT model to disk.